# Lo scenario e i dati

**Azienda:** ToysGroup distribuisce giocattoli tramite una rete di rivenditori in diverse aree geografiche. I dati del Modulo 3 sono esportati in quattro file di formato diverso, forniti dal docente, da leggere, pulire, analizzare e modellare in Python.

### Anagrafiche · CSV ed Excel
- `products.csv`: ProductID, ProductName, Category, ListPrice.
- `regions.xlsx`: RegionID, State, SalesRegion (lo stato appartiene a una regione di vendita).

**Chiavi: ProductID e RegionID.**

### Vendite · JSON
- `sales.json`: SalesID, ProductID, RegionID, ResellerID, SalesDate, Quantity, SalesAmount.
- Una riga per transazione, con valori mancanti in Quantity e SalesAmount.

**Fatti di vendita collegati alle anagrafiche.**

### Rivenditori · CSV
- `resellers.csv`: ResellerID, RegionID, Ordini12m, GiorniUltimoOrdine, ScontoMedio, Reclami.
- Churn = 1 se il rivenditore non riordina da oltre 6 mesi.

**Base per classificazione e clustering.**

---

> **Consegna finale (entro 2 giornate lavorative):** un unico notebook .ipynb eseguibile dall'inizio senza errori, con le celle nell'ordine dei task; ogni cella di codice è preceduta da una cella Markdown con il numero del task e la risposta richiesta. Notebook e dati vanno su GitHub (Task 7).

## Costrutti disponibili

| Concetto | Definizione | Esempio |
|---|---|---|
| List comprehension | costruisce una nuova lista applicando un'espressione a ogni elemento di un iterabile, con un filtro if opzionale | `[p["ListPrice"] for p in products if p["Category"] == "Bikes"]` |
| `.loc[]` e `.iloc[]` | `.loc[]` seleziona righe e colonne per etichetta o maschera booleana; `.iloc[]` per posizione intera | `df.loc[df["ListPrice"] > 30, "ProductName"]`; `df.iloc[10:20, :3]` |
| `groupby()` e `agg()` | divide il DataFrame in gruppi per valore della chiave e applica a ciascun gruppo una o più funzioni di aggregazione | `df.groupby("Category").agg(Fatturato=("SalesAmount", "sum"))` |
| `merge()` e `concat()` | `merge()` affianca le colonne di due DataFrame in base a una chiave comune; `concat()` accoda righe o colonne | `pd.merge(sales, products, on="ProductID", how="left")` |
| Workflow Scikit-learn | sequenza comune a ogni estimator: split dei dati, creazione del modello, fit() sul training, predict() e valutazione sul test | `model = LinearRegression().fit(X_train, y_train)`; `model.score(X_test, y_test)` |

# Task 1 · Python di base sui dati forniti
*Python for Data Science · Parti 1 e 2 · Giorno 1 · circa 1,5 ore*

**Consegna:** sui quattro file forniti dal docente, svolgere i primi calcoli con strutture dati, cicli e funzioni Python, senza metodi di analisi Pandas.

1. leggere `products.csv` e `sales.json` e convertirli con `.to_dict(orient="records")` in liste di dizionari; costruire `regions` come dizionario `{RegionID: (State, SalesRegion)}` e il set delle categorie.
2. con una list comprehension e un costrutto `if/elif/else` assegnare a ogni prodotto la fascia di prezzo: Bassa sotto 20 €, Media fino a 60 €, Alta oltre.
3. scrivere la funzione `fatturato_categoria(vendite, prodotti, categoria)` che con un ciclo `for` somma `SalesAmount` della categoria, salta i valori mancanti e restituisce il totale con `return`.
4. con un ciclo `while` contare le transazioni necessarie a superare 10.000 € di fatturato cumulato; mostrare le prime cinque transazioni con `vendite[:5]`.

**Vincolo:** Pandas si usa solo per leggere i file e convertirli; somme, conteggi e classificazioni si calcolano con costrutti Python.

**Definizione richiesta:** differenza tra lista, tupla, set e dizionario per mutabilità, ordine, duplicati e modalità di accesso (indice o chiave).

> **Criterio di completamento:** `len()` di lista prodotti, dizionario regions e set delle categorie riportata; fascia assegnata a ogni prodotto; fatturato per categoria da `fatturato_categoria()`, da confrontare con il Task 3b.

In [2]:
# Es 1 Punto 1

import pandas as pd

# dataframe
products_df = pd.read_csv("data/products.csv")
sales_df = pd.read_json("data/sales.json")
regions_df = pd.read_excel("data/regions.xlsx")
resellers_df = pd.read_csv("data/resellers.csv")

# liste di dizionari
products = products_df.to_dict(orient="records")
sales = sales_df.to_dict(orient="records")

# dizionario regions
regions = {
    r["RegionID"]: (r["State"], r["SalesRegion"])
    for r in regions_df.to_dict(orient="records")
}

# Set delle categorie
categories = {p["Category"] for p in products}

print("Products:", len(products))
print("Sales:", len(sales))
print("Regions:", len(regions))
print("Categories:", len(categories), categories)

Products: 12
Sales: 2000
Regions: 9
Categories: 3 {'Outdoor', 'Giochi da tavolo', 'Giocattoli'}


In [13]:
# Es 1 punto 2

def price_band(price):
    if price < 20:
        return "Bassa"
    elif price <= 60:
        return "Media"
    else:
        return "Alta"

bands = [price_band(p["ListPrice"]) for p in products]

# creazione di un nuovo dizionario per non modificare quello esistente:

products_banded = [{**p, "PriceBand": price_band(p["ListPrice"])} for p in products]

# verifiche
print([p for p in products if p["ListPrice"] != p["ListPrice"]])
products_banded[:5]


[]


[{'ProductID': 1,
  'ProductName': 'Bici Junior',
  'Category': 'Outdoor',
  'ListPrice': 129.0,
  'PriceBand': 'Alta'},
 {'ProductID': 2,
  'ProductName': 'Bici BMX',
  'Category': 'Outdoor',
  'ListPrice': 189.0,
  'PriceBand': 'Alta'},
 {'ProductID': 3,
  'ProductName': 'Monopattino',
  'Category': 'Outdoor',
  'ListPrice': 79.0,
  'PriceBand': 'Alta'},
 {'ProductID': 4,
  'ProductName': 'Pallone Calcio',
  'Category': 'Outdoor',
  'ListPrice': 19.9,
  'PriceBand': 'Bassa'},
 {'ProductID': 5,
  'ProductName': 'Racchette Beach',
  'Category': 'Outdoor',
  'ListPrice': 14.5,
  'PriceBand': 'Bassa'}]